# Pupper on a skateboard 🛹

Trains the split-stance skateboard tasks (left feet on the deck, right feet
on the floor). Built from `CS123_Pupper_mjlab.ipynb`; the tasks live in
`src/mjlab/tasks/pupper_skate/`.

*Runtime → Change runtime type → **G4 GPU**.* Run all works unattended; or
run cells one at a time. Section 7 prints the verdict: the last line is
`RESULT: PASS` or `RESULT: FAIL`.


## 1. Setup

Done when it prints `mjlab ready`. A red `ERROR: pip's dependency resolver ...`
line about `moviepy` / `decorator` is harmless — Colab's preinstalled `moviepy`
isn't used here. Ignore it.

In [ ]:
REPO_URL = "https://github.com/lai-ethan/cs123_skateboard_pupper.git"  # team repo
BRANCH = "main"

import os

!nvidia-smi -L
if not os.path.exists("/content/pupper-mjlab"):
  !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/pupper-mjlab
%cd /content/pupper-mjlab
!pip install -q -e .

# The running kernel can't see an editable install's path hooks; wire src/
# onto the path so imports work without a runtime restart.
import sys

sys.path.insert(0, "/content/pupper-mjlab/src")
for _m in [k for k in list(sys.modules) if k == "mjlab" or k.startswith("mjlab.")]:
  del sys.modules[_m]
import mjlab.envs  # smoke-test

print("mjlab ready")

## 2. Weights & Biases (optional)

Paste a key from [wandb.ai/authorize](https://wandb.ai/authorize) below, or
store it once as the Colab secret `WANDB_API_KEY` (🔑 in the left bar). This
cell never prompts.

Without a key the run logs to TensorBoard only. You lose the W&B curves and
the automatic `policy.json` upload that `deploy.sh` downloads; the policy
stays in `logs/rsl_rl/` on this runtime, so download it before you
disconnect. Never commit a key.


In [ ]:
WANDB_API_KEY = ""  # optional: paste a key, or set the Colab secret WANDB_API_KEY

import os

if not WANDB_API_KEY:
  try:
    from google.colab import userdata

    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
  except Exception:
    WANDB_API_KEY = ""

if WANDB_API_KEY:
  import wandb

  # A pasted key often carries a stray space or newline; W&B rejects those.
  WANDB_API_KEY = "".join(c for c in WANDB_API_KEY if c.isalnum() or c == "_")

  os.environ["WANDB_API_KEY"] = WANDB_API_KEY
  wandb.login(key=WANDB_API_KEY, relogin=True)
  LOGGER = "wandb"
else:
  LOGGER = "tensorboard"
  print("No W&B key: logging to TensorBoard only. policy.json stays on this runtime.")
print("logger:", LOGGER)

## 3. Choose your task

| Task | What it is |
| --- | --- |
| `Mjlab-Skate-Balance-Pupper-v3` | Hold the split stance on a board that is free to roll. Train this first. |
| `Mjlab-Skate-Ride-Pupper-v3` | Push with the right legs so the board rolls at the commanded speed (0 to 0.5 m/s). |


In [ ]:
TASK = "Mjlab-Skate-Balance-Pupper-v3"
NUM_ENVS = 4096  # halve it if you hit GPU OOM
ITERATIONS = 1500
WATCH_LIVE = False  # True: start the live viewer before training
OPEN_VIEWER = True  # False: skip the interactive viewer at the end

## 4. Reward weights ✏️

Unlike the course tasks, the skate tasks ship with working weights. The
cell below lists every one; change a number to tune it. A term the chosen
task does not have is ignored. Shipped defaults, for reference:

| Term | Balance | Ride | What it does |
| --- | --- | --- | --- |
| `feet_on_deck` | 1.0 | 1.0 | +1 per left foot touching the deck |
| `feet_on_ground` | 0.5 | 0.1 | +1 per right foot touching the floor |
| `deck_feet_on_ground` | -2.0 | -2.0 | left foot on the floor |
| `ground_feet_on_deck` | -2.0 | -2.0 | right foot on the deck |
| `knees_on_deck` | -1.0 | -1.0 | knee resting on the deck |
| `tilt` | 1.0 | 1.0 | hold the stance's 0.09 rad roll, no pitch |
| `base_height` | 1.0 | 1.0 | hold 0.156 m |
| `stance_pose` | 1.0 | 1.0 | joints near the keyframe (ride: left legs only) |
| `board_alignment` | 1.0 | 1.0 | board under the left feet, parallel to the robot |
| `deck_flat` | -5.0 | -5.0 | deck leaning on its trucks |
| `board_still` | -2.0 | n/a | board moving (balance only) |
| `track_board_velocity` | n/a | 3.0 | board speed matches the command |
| `board_yaw_rate` | n/a | -0.5 | board turning |
| `push_stride` | n/a | 0.3 | right feet take planted strides: 0.4 s on the floor, then a 0.2 to 0.5 s swing |
| `push_leg_pose` | n/a | 0.3 | right legs stay loosely near the keyframe |
| `joint_vel_l2` | n/a | -1e-3 | joint speed |
| `joint_speed_limit` | n/a | -0.02 | joint speed above 6 rad/s |
| `torque_saturation` | n/a | -0.5 | motors above 80% of their 3 Nm limit |
| `action_rate_l2` | -0.01 | -0.01 | jerky actions |
| `joint_torques_l2` | -2e-4 | -2e-4 | effort |
| `termination` | -100 | -100 | falling, leaving the deck, tipping the board |


In [ ]:
RIDE = TASK == "Mjlab-Skate-Ride-Pupper-v3"

# These are the shipped defaults. Change a number, then rerun from here.
REWARD_WEIGHTS = {
  "feet_on_deck": 1.0,
  "feet_on_ground": 0.1 if RIDE else 0.5,
  "deck_feet_on_ground": -2.0,
  "ground_feet_on_deck": -2.0,
  "knees_on_deck": -1.0,
  "tilt": 1.0,
  "base_height": 1.0,
  "stance_pose": 1.0,
  "board_alignment": 1.0,
  "deck_flat": -5.0,
  "action_rate_l2": -0.01,
  "joint_torques_l2": -0.0002,
  "termination": -100.0,
  # Balance only (ignored by Ride)
  "board_still": -2.0,
  # Ride only (ignored by Balance)
  "track_board_velocity": 3.0,
  "board_yaw_rate": -0.5,
  "push_stride": 0.3,
  "push_leg_pose": 0.3,
  "joint_vel_l2": -0.001,
  "joint_speed_limit": -0.02,
  "torque_saturation": -0.5,
}

## 5. Domain randomization ✏️

The simulator is not your robot: real motors run weaker or stronger than modeled,
real floors grip differently. Each `(low, high)` range is sampled per episode —
too narrow breaks on hardware, too wide trains a timid crouch.

In [ ]:
KP_MULTIPLIER_RANGE = (0.6, 1.1)  # motor position-gain multiplier (1.0 = as modeled)
KD_MULTIPLIER_RANGE = (0.8, 1.5)  # motor damping-gain multiplier
FRICTION_RANGE = (0.6, 1.4)  # foot-ground friction coefficient

## 6. Train 🚀

*Watch it live* starts a viewer when `WATCH_LIVE = True`; the training cell
after it takes about 9 min for 1,500 iterations on the G4 (about 55 on a T4).
The stop button is safe.

What to watch on W&B (or TensorBoard without a key):

- `Train/mean_episode_length`: 500 is a full 10 s episode.
- `Episode_Reward/feet_on_deck`: 2.0 means both left feet on the deck all episode.
- `Episode_Termination/*`: which termination ends episodes.
- Ride only: `Episode_Reward/track_board_velocity` (3.0 is perfect).


### Watch it live 👀 (optional but recommended — run *before* the training cell)

*This cell does **not** train — it only starts the viewer, and only when
`WATCH_LIVE = True` in section 3.*

Launches a background viewer on this same GPU that follows the run you start
below. Open the printed link in any browser tab — the robot appears once the
first checkpoint lands, about a minute into training (if the page looks empty,
reload it). In the viewer, **Checkpoints → Sync → Use Latest** hot-loads the newest
policy as training writes them; the sliders drive the robot as usual.

**The GPU cost is the open tab, not this cell**: with no tab connected the
viewer pauses itself and training runs at full speed (measured 0% overhead);
while a tab is connected, training runs ~25–30% slower. So peek, then close
the tab — don't leave it open in the background. Re-run this cell before each
new training run so the viewer follows the right one.


In [ ]:
if WATCH_LIVE:
  import subprocess
  import sys
  import time
  from pathlib import Path

  URL_FILE = Path("/content/viewer_url.txt")
  LOG_FILE = Path("/content/viewer.log")
  PID_FILE = Path("/content/viewer_pid.txt")

  # Restart the watcher so it latches onto the run started next, not an old one.
  if PID_FILE.exists():
    subprocess.run(["kill", PID_FILE.read_text().strip()], capture_output=True)
  URL_FILE.unlink(missing_ok=True)
  proc = subprocess.Popen(
    [sys.executable, "notebooks/watch_live.py", TASK, "--url-file", str(URL_FILE)],
    stdout=LOG_FILE.open("w"),
    stderr=subprocess.STDOUT,
  )
  PID_FILE.write_text(str(proc.pid))

  for _ in range(45):
    if URL_FILE.exists():
      print("Watch live here:", URL_FILE.read_text())
      break
    time.sleep(2)
  else:
    print("Viewer didn't come up -- check", LOG_FILE)
  print("Robot appears once the first checkpoint lands (~1 min into training).")

### Train ▶️ — this is the cell that trains

Blocks until training ends.


In [ ]:
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
import mjlab.tasks  # register tasks

env_cfg = load_env_cfg(TASK)
for name, weight in REWARD_WEIGHTS.items():
  if name in env_cfg.rewards:  # terms a task doesn't have are skipped
    env_cfg.rewards[name].weight = weight
env_cfg.events["pd_gains"].params["kp_range"] = KP_MULTIPLIER_RANGE
env_cfg.events["pd_gains"].params["kd_range"] = KD_MULTIPLIER_RANGE
env_cfg.events["foot_friction"].params["ranges"] = FRICTION_RANGE
env_cfg.scene.num_envs = NUM_ENVS

agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.logger = LOGGER

launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg))

## 7. Did it pass?

Averages the last 20 iterations of the newest run and checks them against
the pass criteria in `skateboard/tools/run_summary.py`. The last line is
`RESULT: PASS` or `RESULT: FAIL`.


In [ ]:
!python skateboard/tools/run_summary.py {TASK}

### Gait check (ride)

Curves can pass while the gait is one the robot cannot do. This rolls out the
newest checkpoint at 0.4 m/s and checks joint speed, torque, how the right feet
step and that the board rolls at half the command or faster. The last line
is `GAIT: PASS` or `GAIT: FAIL`. Watch the video too before trusting a pass.


In [ ]:
from pathlib import Path

if RIDE:
  runs = sorted(Path("logs/rsl_rl").glob("*/*"), key=lambda p: p.stat().st_mtime)
  assert runs, "No runs yet -- train first."
  ckpts = sorted(
    runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1])
  )
  ckpt = ckpts[-1]
  print("Gait check on:", ckpt)
  !python skateboard/tools/gait_report.py {TASK} {ckpt} --speed 0.4
else:
  print("Gait check is for the ride task only.")

## 8. Watch it 🕹️

Loads your newest checkpoint into an interactive 3D viewer (public link, works
from Colab). Drive the robot with the sliders. Skipped when
`OPEN_VIEWER = False`; when it runs, the cell keeps running until you stop it.

This only finds checkpoints on *this* runtime's disk. Runtime disconnected? Your
checkpoints are still on W&B — see "Lost your runtime?" in the lab 4 instructions
(step 4) for a cell that replays any run by its W&B run id.

In [ ]:
if OPEN_VIEWER:
  from dataclasses import asdict
  from pathlib import Path

  import viser
  from mjlab.envs import ManagerBasedRlEnv
  from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
  from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
  from mjlab.viewer import ViserPlayViewer

  runs = sorted(Path("logs/rsl_rl").glob("*/*"), key=lambda p: p.stat().st_mtime)
  assert runs, "No runs yet -- train first."
  ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
  print("Using checkpoint:", ckpts[-1])

  device = "cuda:0"
  env_cfg = load_env_cfg(TASK, play=True)
  env_cfg.scene.num_envs = 1
  agent_cfg = load_rl_cfg(TASK)
  env = RslRlVecEnvWrapper(
    ManagerBasedRlEnv(cfg=env_cfg, device=device), clip_actions=agent_cfg.clip_actions
  )
  runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
  runner = runner_cls(env, asdict(agent_cfg), device=device)
  runner.load(
    str(ckpts[-1]), load_cfg={"actor": True}, strict=True, map_location=device
  )

  server = viser.ViserServer(label="pupper", share=True)
  print("Open the viewer here:", server.request_share_url())
  ViserPlayViewer(
    env, runner.get_inference_policy(device=device), viser_server=server
  ).run()

## 9. Deploy

With W&B, `policy.json` uploads to the run when training ends; without it,
download `policy.json` from the run folder in `logs/rsl_rl/`. **Do not run a
skate policy on the robot without a TA sign-off**, and do the powered-off
fit test on the real board first (see `skateboard/PLAN.md`, tasks 10 and 12).
